# HalDetect — Non-routed baseline (RQ2) on Colab

**Why Colab:** the Feb–Apr session handoff already designated this verifier as the
Colab/Kaggle-GPU one — *"most verifiers can run locally; the RELATIONAL verifier
(Sentence-BERT + bart-large-mnli) is the one suited to Colab/Kaggle GPU."* The
non-routed baseline **is** that verifier, run over every claim instead of only the
RELATIONAL ones.

It also sidesteps the `zsh: bus error` seen locally, which killed BART on its first
forward pass even with an 82-character premise and a single thread.

## Two things that matter more than speed

**1. The cache IS the reproducibility guarantee.** Gemini query generation is
non-deterministic, and the project rule is that thesis numbers come from a *single
cached run*, never an average. Cell 2 mounts Drive and keeps `nonrouted_cache.json`
there. **If you skip it, a disconnect costs you the run's reproducibility, not just
its progress.**

**2. Pass `--device 0`.** `phase3d` hardcodes `device=-1`, so on a GPU runtime BART
would silently run on the CPU. `--device 0` implies `--safe-load`, which is how the
override is applied. Set **Runtime → Change runtime type → T4 GPU** first.


## 1. Upload the code + data

Run this **on your Mac** first to build a minimal zip (a few MB — it deliberately
excludes `.venv`):

```bash
cd "/Users/sahibabc/8th Semester Materials/ThesisCode_copy"
zip -r haldetect_min.zip \
  current/*.py \
  current/haldetect_claims_b3_gold.csv \
  "current/haldetect_phase2_goldlabeled (3).csv" \
  current/relational_negatives_batch3_merged.csv \
  current/relational_retrieval_cache.json \
  current/relational_nli_cache.json \
  archive/scripts/relational_verifier_phase3d.py \
  archive/csvs/haldetect_claims_20260524_045439_2.csv
```

**Both cache files matter, and the second one especially.**

- `relational_retrieval_cache.json` (1.7 MB) holds the fetched Wikipedia page text
  **and** the `llmquery__` entries. Those entries are what keep Gemini query
  generation deterministic for claims already seen — without them the same claim can
  resolve to a different page on a later run and flip its verdict.
- `relational_nli_cache.json` (25 KB) holds the NLI scores. Every cached
  (claim, passage) pair **skips the BART call entirely**: `run_nli_on_passages`
  checks the cache before calling the model, and `get_nli()` is lazy, so a fully
  cached claim never loads BART at all.

There is no `relational_cache.json` in this workspace — that was Phase 3d's single
combined cache. Phase 3e split it into the pair above, which is why the earlier zip
command warned `name not matched`. Verified key-format compatibility: `nli__`,
`text__` and `llmquery__` are byte-identical between the two generations and
transfer cleanly; only Phase 3e's `titles__…__3` multi-candidate keys don't, and
re-running a Wikipedia search is cheap.

Then upload the zip below.

In [ ]:
from google.colab import files
import zipfile, os, pathlib

up = files.upload()                      # pick haldetect_min.zip
name = next(iter(up))
root = '/content/haldetect'
os.makedirs(root, exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall(root)

for p in sorted(pathlib.Path(root).rglob('*')):
    if p.is_file():
        print(f'{p.stat().st_size:>10,}  {p.relative_to(root)}')

## 2. Persist the cache to Drive (do not skip — see the note at the top)

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
PERSIST = '/content/drive/MyDrive/haldetect'
os.makedirs(PERSIST, exist_ok=True)

LIVE = '/content/haldetect/current/nonrouted_cache.json'
SAVED = os.path.join(PERSIST, 'nonrouted_cache.json')

# Restore a previous run's cache if there is one. Otherwise the script builds it
# by MERGING relational_retrieval_cache.json + relational_nli_cache.json on first
# use, and reports the per-prefix counts so you can confirm the NLI half landed.
if os.path.exists(SAVED):
    shutil.copy2(SAVED, LIVE)
    print(f'restored cache from Drive ({os.path.getsize(SAVED):,} bytes)')
else:
    print('no saved cache yet - will be merged from the seed caches on first run')

def save_cache():
    """Call after every run. Cheap, and the run is not reproducible without it."""
    if os.path.exists(LIVE):
        shutil.copy2(LIVE, SAVED)
        print(f'cache -> Drive ({os.path.getsize(LIVE):,} bytes)')
    else:
        print('!! no live cache found - nothing saved')

## 3. Dependencies and the Gemini key

`gemini_key.py` resolves in the order env var → key file → **Colab `userdata`**, so
the cleanest option here is Colab Secrets: click the 🔑 in the left sidebar, add
`GEMINI_API_KEY`, and enable notebook access. Nothing else to configure — and the
key never gets written into the notebook.

In [ ]:
!pip -q install "transformers>=4.40" sentence-transformers requests

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('\n!! No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
    print('   You can still proceed on CPU with --device -1, just slower.')

In [ ]:
%cd /content/haldetect
!python current/nonrouted_baseline.py --doctor

## 4. Smoke tests, in order

Each one is cheap and rules out a different failure. Don't skip to the full run.

In [ ]:
# Does BART survive a forward pass here at all? (This is what died locally.)
!python current/nonrouted_baseline.py --nli-check --device 0

In [ ]:
# Plumbing only, no network. Expect 356 claims and the type mix
# CITATION 29, ENTITY_ATTRIBUTE 43, QUANTITATIVE 45, RELATIONAL 174, TEMPORAL 65.
!python current/nonrouted_baseline.py --dry-run

In [ ]:
# 10 claims, stratified 2 per type, end to end.
!python current/nonrouted_baseline.py --limit 10 --device 0
save_cache()

## 5. The actual experiment

All 356 labeled claims. Checkpoints the CSV every 10 claims, so a disconnect costs
time but not work — rerun and cached claims are skipped.

**What to expect, so the result isn't misread:** the baseline should look *good* on
RELATIONAL, because there it is the same code path the routed system uses. The
informative rows are TEMPORAL / ENTITY_ATTRIBUTE / QUANTITATIVE / CITATION, where
routing sends claims elsewhere. **If there is no gap, that is also a result** and it
gets reported rather than tuned away.

In [ ]:
!python current/nonrouted_baseline.py --device 0 2>&1 | tail -120
save_cache()

## 6. Bring the results home

Copy to Drive **and** download. The CSV is the artifact the thesis tables are built
from; the cache is what makes the numbers reproducible.

In [ ]:
import shutil, os
from google.colab import files

CSV = '/content/haldetect/current/nonrouted_baseline_results.csv'
save_cache()
if os.path.exists(CSV):
    shutil.copy2(CSV, os.path.join(PERSIST, 'nonrouted_baseline_results.csv'))
    print('copied to Drive')
    files.download(CSV)
else:
    print('no results CSV — the run did not get far enough')

## 7. After it finishes

Put `nonrouted_baseline_results.csv` back into `current/` on your Mac, then:

- Read the **two** printed report blocks together — all-labeled and b1+b3-only. Phase 2's
  109 verdicts were AI-filled then AI-corrected, so if the two diverge, the pooled figure
  leans on unreviewed labels and is not quotable.
- Check the coverage-accuracy tradeoff table (3d-style vs 3e-style) — that is the
  RELATIONAL reporting decision, settled as *report both*.
- Compare the per-type table against the routed numbers, and re-measure the routed side
  with the **same** `--max-passage-chars` value so both halves see identical premises.
- Note the premise cap in the thesis method section. `chunk_passages()` had no maximum
  length, so long Wikipedia paragraphs previously ran past bart-large-mnli's 1024-token
  window on **both** sides of the ablation.


## 8. Second run — the negatives (this is what unblocks detection numbers)

The main run above measures **coverage and accuracy on mostly-TRUE claims**. It
cannot measure *detection*, because 88% of the gold set is TRUE and only 13 claims
are naturally FALSE.

`current/synthetic_negatives.csv` holds 85 deliberately-corrupted claims (TEMPORAL
31, CITATION 25, QUANTITATIVE 24, ENTITY_ATTRIBUTE 5), each recording what was
changed and by how much. Running the baseline over them does two things:

1. gives the **non-routed baseline** a false-alarm / detection rate on negatives;
2. fills the tier-2 cache the **hybrid** needs. `temporal_ea_verifier.py --hybrid`
   looks tier-2 verdicts up by `claim_id`, and the `SYN_*` ids were never in the
   main run — so right now the hybrid has a coverage number and **no** detection
   number. This run is what closes that.

Note `--out`: without it this would overwrite `nonrouted_baseline_results.csv` and
destroy the main result.

In [ ]:
!python current/nonrouted_baseline.py \
    --claims-csv current/synthetic_negatives.csv \
    --out current/nonrouted_negatives_results.csv \
    2>&1 | tee /content/haldetect/current/negatives_run.log | tail -60

save_cache()

In [ ]:
import shutil, os
from google.colab import files

# Bring BOTH result files home. The negatives file is the one that unblocks the
# hybrid's detection number; the cache is what makes any of it reproducible.
for f in ("nonrouted_baseline_results.csv",
          "nonrouted_negatives_results.csv",
          "nonrouted_cache.json"):
    src = f"/content/haldetect/current/{f}"
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(PERSIST, f))
        print(f"{os.path.getsize(src):>10,}  {f}  -> Drive")
        files.download(src)
    else:
        print(f"{'MISSING':>10}  {f}")